In [4]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from urllib.parse import urlparse

# === Load your GitHub token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN_3")
if not GITHUB_TOKEN:
    raise ValueError("❌ GITHUB_TOKEN_3 not found in All_Tokens.env")

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# === Paths ===
BASE_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd"
INPUT_CSV = os.path.join(BASE_DIR, "GitHub_Android_Projects_URLs_test.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "Commits")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Fetch commits ===
def get_commit_history(owner, repo, max_commits=500):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    commits = []
    page = 1

    while len(commits) < max_commits:
        paged_url = f"{url}?per_page=100&page={page}"
        response = requests.get(paged_url, headers=headers)

        if response.status_code != 200:
            print(f"❌ Error fetching {owner}/{repo}: {response.status_code}")
            break

        data = response.json()
        if not data:
            break  # No more commits

        for commit in data:
            commit_data = commit.get("commit", {})
            author = commit_data.get("author", {})
            commits.append({
                "sha": commit.get("sha"),
                "author_name": author.get("name"),
                "author_email": author.get("email"),
                "date": author.get("date"),
                "message": commit_data.get("message"),
                "html_url": commit.get("html_url")
            })

            if len(commits) >= max_commits:
                break

        page += 1

    return pd.DataFrame(commits)

# === Process each repo from CSV ===
df_urls = pd.read_csv(INPUT_CSV)
for url in df_urls["html_url"].dropna().unique():

    try:
        path_parts = urlparse(url).path.strip("/").split("/")
        if len(path_parts) != 2:
            print(f"⚠️ Skipping malformed URL: {url}")
            continue
        owner, repo = path_parts
        full_name = f"{owner}.{repo}"
        print(f"🔄 Fetching commits for: {full_name}")

        df_commits = get_commit_history(owner, repo, max_commits=500)
        if not df_commits.empty:
            output_file = os.path.join(OUTPUT_DIR, f"{full_name}.commits.csv")
            df_commits.to_csv(output_file, index=False)
            print(f"✅ Saved to {output_file}")
        else:
            print(f"⚠️ No commits found for {full_name}")

    except Exception as e:
        print(f"❌ Failed to process {url}: {e}")


🔄 Fetching commits for: AChep.15puzzle
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\AChep.15puzzle.commits.csv
🔄 Fetching commits for: italankin.15Puzzle
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\italankin.15Puzzle.commits.csv
🔄 Fetching commits for: Abdallah-Sobehy.15puzzle
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\Abdallah-Sobehy.15puzzle.commits.csv
🔄 Fetching commits for: hapramp.1Rramp-Android
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\hapramp.1Rramp-Android.commits.csv
🔄 Fetching commits for: CSID-DGU.2021-1-OSSP2-Barcode-8
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\CSID-DGU.2021-1-OSSP2-Barcode-8.commits.csv
🔄 Fetching commits for: pknu-wap.2022_2_WAP_APP_TEAM1
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\pknu-wap.2022_2_WAP_APP_TEAM1.commits.csv
🔄 Fetching commits for: uberspot.2048-android
✅ Saved to F:\Android_Mobile_App\AndroidProject_3rd\Commits\uberspot.2048-android.commits.